In [1]:
#pip install requests beautifulsoup4 pandas lxml html5lib 
#pip install --upgrade certifi

Pour la suite du developpement, 
je dois recuperer la date et l'heure de l'actualisaéton à chaque fois
cette information sera donc sauvegardé dans la table des actions fusionnées
l'idée derrière cette modification c'est d'incrementer les données sur les données precedemment sauvegarder à chaque fois.

Ensuite une reflexion doit etre mené pour la sauvegarde... est ce que on passera par sql ou une sauvegarde classique dans un csv qui sera à chaque fois incrementer( voir limite potentielle de cette methode)

Une fois la partie data achevée, nous pourrons nous focaliser sur la recuperation de l'historique sur le github de koffi(https://github.com/Fredysessie/brvm-data-public/blob/main/data/LNBB/LNBB.daily.csv)

Une fois cette etape achevée, nous pourrons alors penser à la modelisation des données hitoire d'obtenir les indicateurs clés de notre analyse sur la brvm (Reflexion à mener sur les indicateurs et leurs pertinence)

In [2]:
import pandas as pd
import requests
import urllib3
from io import StringIO
from typing import List, Optional
import re
from datetime import datetime

# Désactivation de l'avertissement SSL (si nécessaire pour BRVM)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def clean_brvm_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Nettoie automatiquement toutes les colonnes d'un DataFrame de la BRVM.
    Gère les espaces, les espaces insécables, les virgules et les pourcentages.
    """
    df = df.copy() # Évite les warnings SettingWithCopy
    df.columns = df.columns.str.strip()
    
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
            
        col_str = df[col].astype(str)
        is_pct = '%' in col or col_str.str.contains('%', na=False).any()
        
        # Chaînage optimisé : Suppression des espaces (\s et \xa0) et remplacement de la virgule
        cleaned = col_str.str.replace(r'[\s\xa0]+', '', regex=True).str.replace(',', '.', regex=False)
        
        if is_pct:
            cleaned = cleaned.str.replace('%', '', regex=False)
            df[col] = pd.to_numeric(cleaned, errors='coerce') / 100
        else:
            temp_numeric = pd.to_numeric(cleaned, errors='coerce')
            
            # Vérification de sécurité (heuristique des NaN)
            original_empty = cleaned.isin(['nan', 'None', '', 'NaN']).sum()
            new_nan = temp_numeric.isna().sum()
            
            if new_nan == original_empty and temp_numeric.notna().sum() > 0:
                df[col] = temp_numeric
                
    return df



def fetch_brvm_tables(url: str, session: requests.Session) -> List[pd.DataFrame]:
    """
    Récupère de manière sécurisée les tables HTML d'une URL donnée.
    """
    print(f"--- Extraction en cours : {url} ---")
    headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}
    
    try:
        response = session.get(url, headers=headers, verify=False, timeout=15)
        response.raise_for_status()
        return pd.read_html(StringIO(response.text))
    except requests.exceptions.RequestException as e:
        print(f"❌ Erreur réseau lors de l'accès à {url} : {e}")
        return []
    except ValueError:
        print(f"❌ Aucune table HTML trouvée sur {url}.")
        return []
    

    
    
def refresh_date(url: str, session: requests.Session):
    """
    Récupère la date de dernière mise à jour sur le site de la BRVM 
    et la retourne sous forme de timestamp UNIX.
    """
    mois_fr = {
        'janvier': 1, 'février': 2, 'mars': 3, 'avril': 4, 'mai': 5, 'juin': 6,
        'juillet': 7, 'août': 8, 'septembre': 9, 'octobre': 10, 'novembre': 11, 'décembre': 12
    }
    
    try:
        # 1. On exécute la requête pour obtenir le HTML
        # On met un entête classique pour éviter d'être bloqué
        headers = {"User-Agent": "Mozilla/5.0"}
        response = session.get(url, headers=headers, verify=False, timeout=15)
        response.raise_for_status()

        # 2. On cherche le texte situé juste après "Dernière mise à jour :"
        match_maj = re.search(r'Dernière mise à jour\s*:\s*([^\n<]+)', response.text)

        if match_maj:
            date_actualisation = match_maj.group(1).strip() 
            
            # 3. On applique la regex de conversion sur la date trouvée
            match_dt = re.search(r'(\d+)\s+([a-zA-Zéû]+)[,\s]+(\d{4})\s*-\s+(\d{2}):(\d{2})', date_actualisation.lower())
            
            if match_dt:
                dt = datetime(
                    int(match_dt.group(3)),              # Année
                    mois_fr.get(match_dt.group(2), 1),   # Mois
                    int(match_dt.group(1)),              # Jour
                    int(match_dt.group(4)),              # Heure
                    int(match_dt.group(5))               # Minute
                )
                
                print(f"✅ Date refresh : {dt}")
                return dt

    except Exception as e:
        print(f"❌ Erreur lors de l'extraction de la date : {e}")

    # Retourne None si la date n'est pas trouvée ou si la requête échoue
    return None



def main():
    # 1. Utilisation d'une session pour accélérer les requêtes multiples sur le même domaine
    with requests.Session() as session:
        url_actions = "https://www.brvm.org/fr/cours-actions/0"
        tables_actions = fetch_brvm_tables(url_actions, session)
        
        url_volumes = "https://www.brvm.org/fr/volumes/0"
        tables_volumes = fetch_brvm_tables(url_volumes, session)

        date_refresh = refresh_date(url_actions, session)

    # 2. Vérification de l'intégrité des données extraites
    if len(tables_actions) < 4 or len(tables_volumes) < 4:
        print("❌ Extraction incomplète. Arrêt du script.")
        return None

    # 3. Nettoyage ciblé uniquement sur les tables nécessaires
    print("--- Nettoyage des données ---")
    df_actions = clean_brvm_dataframe(tables_actions[3])
    volume = clean_brvm_dataframe(tables_volumes[3])

    # 4. Fusion des DataFrames
    print("--- Fusion des données ---")
    actions = df_actions.merge(
        volume, 
        left_on=["Symbole", "Nom"], 
        right_on=["Code obligation", "Nom"], 
        how="left"
    )

    # Filtrage sécurisé des colonnes (évite une KeyError si le site BRVM change une colonne)
    colonnes_voulues = [
        'Symbole', 'Nom', 'Volume', 'Cours veille (FCFA)', 'Cours Ouverture (FCFA)', 
        'Cours Clôture (FCFA)', 'Variation (%)', 'Valeur échangée', 'PER', 
        'Pourcentage de la valeur globale échangée'
    ]
    colonnes_existantes = [col for col in colonnes_voulues if col in actions.columns]
    actions = actions[colonnes_existantes]

    # 5. Optimisation : Remplacement du lambda par une opération vectorisée Pandas
    if 'PER' in actions.columns:
        actions['PER']=actions['PER'].apply(lambda x: int(x)/100 if x.isdigit() else x)

    print("✅ Traitement terminé avec succès !")

    #Intrsion date
    actions['Date Refresh'] = date_refresh
    return actions

# --- Exécution ---
if __name__ == "__main__":
    df_final = main()
    # Si tu veux afficher un aperçu :
    # if df_final is not None:
    #     print(df_final.head())

--- Extraction en cours : https://www.brvm.org/fr/cours-actions/0 ---
--- Extraction en cours : https://www.brvm.org/fr/volumes/0 ---
✅ Date refresh : 2026-05-11 13:04:00
--- Nettoyage des données ---
--- Fusion des données ---
✅ Traitement terminé avec succès !


In [3]:
df_final

,Symbole,Nom,Volume,Cours veille (FCFA),Cours Ouverture (FCFA),Cours Clôture (FCFA),Variation (%),Valeur échangée,PER,Pourcentage de la valeur globale échangée,Date Refresh
0,ABJC,SERVAIR ABIDJAN COTE D'IVOIRE,7762,2955,2995,2960,-50,22924200,24.23,249,2026-05-11 13:04:00
1,BICB,BANQUE INTERNATIONALE POUR L’INDUSTRIE ET LE C...,1541,5225,5165,5200,-48,7980845,9.95,87,2026-05-11 13:04:00
2,BICC,BICI COTE D'IVOIRE,539,26000,26095,26095,37,14044240,11.87,153,2026-05-11 13:04:00
3,BNBC,BERNABE COTE D'IVOIRE,845,1390,1345,1390,692,1121935,412.55,12,2026-05-11 13:04:00
4,BOAB,BANK OF AFRICA BENIN,11991,9400,9400,9390,-11,112860910,18.96,1227,2026-05-11 13:04:00
5,BOABF,BANK OF AFRICA BURKINA FASO,3372,5400,5400,5410,19,18217515,12.34,198,2026-05-11 13:04:00
6,BOAC,BANK OF AFRICA COTE D'IVOIRE,3457,8695,8700,8600,-121,29774835,10.85,324,2026-05-11 13:04:00
7,BOAM,BANK OF AFRICA MALI,4011,4700,4700,4680,-43,18810730,11.64,205,2026-05-11 13:04:00
8,BOAN,BANK OF AFRICA NIGER,7281,3485,3480,3480,-14,25324145,177.12,275,2026-05-11 13:04:00
9,BOAS,BANK OF AFRICA SENEGAL,6130,7390,7400,7425,47,45536170,12.15,495,2026-05-11 13:04:00


In [4]:
import pandas as pd
import requests
import urllib3
from io import StringIO
from typing import List, Tuple, Optional
import re
from datetime import datetime

# Désactivation de l'avertissement SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Constante globale pour éviter la répétition
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}

def clean_brvm_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Nettoie automatiquement toutes les colonnes d'un DataFrame de la BRVM.
    Gère les espaces, les espaces insécables, les virgules et les pourcentages.
    """
    df = df.copy() 
    df.columns = df.columns.str.strip()
    
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
            
        col_str = df[col].astype(str)
        is_pct = '%' in col or col_str.str.contains('%', na=False).any()
        
        cleaned = col_str.str.replace(r'[\s\xa0]+', '', regex=True).str.replace(',', '.', regex=False)
        
        if is_pct:
            cleaned = cleaned.str.replace('%', '', regex=False)
            df[col] = pd.to_numeric(cleaned, errors='coerce') / 100
        else:
            temp_numeric = pd.to_numeric(cleaned, errors='coerce')
            
            original_empty = cleaned.isin(['nan', 'None', '', 'NaN']).sum()
            new_nan = temp_numeric.isna().sum()
            
            if new_nan == original_empty and temp_numeric.notna().sum() > 0:
                df[col] = temp_numeric
                
    return df


def fetch_brvm_data(url: str, session: requests.Session) -> Tuple[List[pd.DataFrame], str]:
    """
    Récupère de manière sécurisée les tables HTML et le texte brut d'une URL donnée.
    """
    print(f"--- Extraction en cours : {url} ---")
    
    try:
        response = session.get(url, headers=HEADERS, verify=False, timeout=15)
        response.raise_for_status()
        html_text = response.text
        # On retourne les tables ET le HTML pour éviter de refaire une requête plus tard
        return pd.read_html(StringIO(html_text)), html_text
    except Exception as e:
        print(f"❌ Erreur réseau lors de l'accès à {url} : {e}")
        return [], ""


def refresh_date(html_text: str) -> Optional[datetime]:
    """
    Extrait la date de dernière mise à jour directement depuis une chaîne HTML.
    Aucune requête réseau n'est effectuée ici.
    """
    if not html_text:
        return None
        
    mois_fr = {
        'janvier': 1, 'février': 2, 'mars': 3, 'avril': 4, 'mai': 5, 'juin': 6,
        'juillet': 7, 'août': 8, 'septembre': 9, 'octobre': 10, 'novembre': 11, 'décembre': 12
    }
    
    try:
        match_maj = re.search(r'Dernière mise à jour\s*:\s*([^\n<]+)', html_text)

        if match_maj:
            date_actualisation = match_maj.group(1).strip() 
            match_dt = re.search(r'(\d+)\s+([a-zA-Zéû]+)[,\s]+(\d{4})\s*-\s+(\d{2}):(\d{2})', date_actualisation.lower())
            
            if match_dt:
                dt = datetime(
                    int(match_dt.group(3)),
                    mois_fr.get(match_dt.group(2), 1),
                    int(match_dt.group(1)),
                    int(match_dt.group(4)),
                    int(match_dt.group(5))
                )
                print(f"✅ Date refresh trouvée : {dt}")
                return dt

    except Exception as e:
        print(f"❌ Erreur lors de l'extraction de la date : {e}")

    return None


def main():
    # 1. Extraction (Session unique maintenue ouverte)
    with requests.Session() as session:
        url_actions = "https://www.brvm.org/fr/cours-actions/0"
        # On récupère les tables ET le code HTML de la page actions
        tables_actions, html_actions = fetch_brvm_data(url_actions, session)
        
        url_volumes = "https://www.brvm.org/fr/volumes/0"
        # Le HTML des volumes n'est pas nécessaire, on l'ignore avec "_"
        tables_volumes, _ = fetch_brvm_data(url_volumes, session)

        # 2. Récupération de la date SANS nouvelle requête (on utilise le HTML déjà téléchargé)
        date_refresh = refresh_date(html_actions)

    # 3. Vérification de l'intégrité
    if len(tables_actions) < 4 or len(tables_volumes) < 4:
        print("❌ Extraction incomplète. Arrêt du script.")
        return None

    # 4. Nettoyage
    print("--- Nettoyage des données ---")
    df_actions = clean_brvm_dataframe(tables_actions[3])
    volume = clean_brvm_dataframe(tables_volumes[3])

    # 5. Fusion
    print("--- Fusion des données ---")
    actions = df_actions.merge(
        volume, 
        left_on=["Symbole", "Nom"], 
        right_on=["Code obligation", "Nom"], 
        how="left"
    )

    # 6. Filtrage sécurisé
    colonnes_voulues = [
        'Symbole', 'Nom', 'Volume', 'Cours veille (FCFA)', 'Cours Ouverture (FCFA)', 
        'Cours Clôture (FCFA)', 'Variation (%)', 'Valeur échangée', 'PER', 
        'Pourcentage de la valeur globale échangée'
    ]
    colonnes_existantes = [col for col in colonnes_voulues if col in actions.columns]
    actions = actions[colonnes_existantes]

    # 7. process PER
    if 'PER' in actions.columns:
        actions['PER']=actions['PER'].apply(lambda x: int(x)/100 if x.isdigit() else x)

    # 8. Insertion de la date
    actions['Date Refresh'] = date_refresh
    
    print("✅ Traitement terminé avec succès !")
    return actions


if __name__ == "__main__":
    df_final = main()

--- Extraction en cours : https://www.brvm.org/fr/cours-actions/0 ---
--- Extraction en cours : https://www.brvm.org/fr/volumes/0 ---
✅ Date refresh trouvée : 2026-05-12 19:15:00
--- Nettoyage des données ---
--- Fusion des données ---
✅ Traitement terminé avec succès !


In [5]:
df_final

,Symbole,Nom,Volume,Cours veille (FCFA),Cours Ouverture (FCFA),Cours Clôture (FCFA),Variation (%),Valeur échangée,PER,Pourcentage de la valeur globale échangée,Date Refresh
0,ABJC,SERVAIR ABIDJAN COTE D'IVOIRE,7588,2985,2995,2980,-17,22618910,24.47,154,2026-05-12 19:15:00
1,BICB,BANQUE INTERNATIONALE POUR L’INDUSTRIE ET LE C...,1578,5170,5200,5200,58,8201500,9.84,56,2026-05-12 19:15:00
2,BICC,BICI COTE D'IVOIRE,561,26200,26200,26300,38,14757815,11.96,100,2026-05-12 19:15:00
3,BNBC,BERNABE COTE D'IVOIRE,7896,1490,1415,1495,717,11638415,442.23,79,2026-05-12 19:15:00
4,BOAB,BANK OF AFRICA BENIN,11117,9405,9405,9420,16,104616915,18.97,712,2026-05-12 19:15:00
5,BOABF,BANK OF AFRICA BURKINA FASO,5219,5410,5400,5410,0,28188865,12.36,192,2026-05-12 19:15:00
6,BOAC,BANK OF AFRICA COTE D'IVOIRE,7396,8650,8655,8585,-75,63520075,10.8,432,2026-05-12 19:15:00
7,BOAM,BANK OF AFRICA MALI,4155,4685,4695,4680,-11,19491500,11.61,133,2026-05-12 19:15:00
8,BOAN,BANK OF AFRICA NIGER,5724,3450,3445,3440,-29,19710885,175.34,134,2026-05-12 19:15:00
9,BOAS,BANK OF AFRICA SENEGAL,8857,7450,7460,7500,67,66160735,12.24,450,2026-05-12 19:15:00


In [ ]:
import os
from supabase import create_client, Client
# ... (Tout ton code de nettoyage précédent reste identique) ...

    # 8. Insertion de la date
    actions['Date Refresh'] = date_refresh
    
    print("✅ Traitement terminé avec succès !")

    # --- NOUVEAU : PREPARATION POUR SUPABASE ---
    print("--- Envoi des données vers Supabase ---")
    
    # 1. Renommer les colonnes pour qu'elles correspondent exactement à la table SQL
    colonnes_mapping = {
        'Symbole': 'symbole',
        'Nom': 'nom',
        'Volume': 'volume',
        'Cours veille (FCFA)': 'cours_veille',
        'Cours Ouverture (FCFA)': 'cours_ouverture',
        'Cours Clôture (FCFA)': 'cours_cloture',
        'Variation (%)': 'variation',
        'Valeur échangée': 'valeur_echangee',
        'PER': 'per',
        'Pourcentage de la valeur globale échangée': 'pourcentage_valeur',
        'Date Refresh': 'date_refresh'
    }
    actions = actions.rename(columns=colonnes_mapping)

    # 2. Convertir la date datetime en format texte ISO (Requis par les bases de données)
    # On vérifie que la date existe bien
    if date_refresh:
        actions['date_refresh'] = actions['date_refresh'].dt.isoformat()
    
    # 3. Remplacer les éventuels NaN par None (JSON/Supabase n'aime pas les NaN de Pandas)
    actions = actions.where(pd.notnull(actions), None)

    # 4. Convertir le DataFrame en liste de dictionnaires
    data_to_insert = actions.to_dict(orient='records')

    # 5. Connexion à Supabase
    # REMPLACE CES VALEURS PAR TES CLES SUPABASE
    SUPABASE_URL = "https://TON_PROJET.supabase.co"
    SUPABASE_KEY = "TA_CLE_ANON_PUBLIQUE"
    
    try:
        supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
        
        # On fait un "upsert" : si la donnée existe déjà (même symbole, même date), il l'ignore
        response = supabase.table("historique_cours").upsert(data_to_insert).execute()
        print(f"✅ {len(data_to_insert)} lignes envoyées à Supabase avec succès !")
    except Exception as e:
        print(f"❌ Erreur lors de l'envoi à Supabase : {e}")

    return actions

if __name__ == "__main__":
    df_final = main()

In [3]:
pip install python-dotenv

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 723.0 kB/s  0:00:10 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 2.7 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━  8/13 [plotly]s]s]  WARNING: The script plotly_get_chrome is installed in '/Library/Frameworks/Python.framework/Versions/3.13/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━  9/13 [jinja2]  WARNING: The script flask is installed in '/Library/Frameworks/Python.framework/Versions/3.13/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 12/13 [dash]  WARNING: The scripts dash-generate-components, dash-update-components, plot

In [7]:
pip install plotly


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:



# --- 2. INITIALISATION DE L'APPLICATION DASH ---
app = Dash(__name__)
app.title = "BRVM Analytics Pro"

# --- 3. MISE EN PAGE (HTML structuré en Python) ---
app.layout = html.Div(style={'fontFamily': 'Arial, sans-serif', 'padding': '20px', 'backgroundColor': '#f8f9fa'}, children=[
    
    # En-tête
    html.H1("📈 Terminal BRVM Analytics", style={'textAlign': 'center', 'color': '#2c3e50'}),
    html.P("Plateforme d'Intelligence Financière", style={'textAlign': 'center', 'color': '#7f8c8d'}),
    
    # Contrôles (Menu déroulant)
    html.Div(style={'width': '300px', 'margin': '0 auto', 'paddingBottom': '30px'}, children=[
        html.Label("Sélectionnez une action :", style={'fontWeight': 'bold'}),
        dcc.Dropdown(
            id='dropdown-action',
            options=[{'label': sym, 'value': sym} for sym in symboles_disponibles],
            value=symboles_disponibles[0] if symboles_disponibles else None,
            clearable=False
        )
    ]),

    # Cartes KPI (Chiffres clés)
    html.Div(style={'display': 'flex', 'justifyContent': 'space-around', 'marginBottom': '30px'}, children=[
        html.Div(style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '10px', 'boxShadow': '0 4px 6px rgba(0,0,0,0.1)', 'textAlign': 'center', 'width': '20%'}, children=[
            html.H4("Cours Actuel", style={'margin': '0', 'color': '#7f8c8d'}),
            html.H2(id='kpi-cours', style={'margin': '10px 0', 'color': '#2c3e50'})
        ]),
        html.Div(style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '10px', 'boxShadow': '0 4px 6px rgba(0,0,0,0.1)', 'textAlign': 'center', 'width': '20%'}, children=[
            html.H4("Volume", style={'margin': '0', 'color': '#7f8c8d'}),
            html.H2(id='kpi-volume', style={'margin': '10px 0', 'color': '#2c3e50'})
        ]),
        html.Div(style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '10px', 'boxShadow': '0 4px 6px rgba(0,0,0,0.1)', 'textAlign': 'center', 'width': '20%'}, children=[
            html.H4("RSI (14j)", style={'margin': '0', 'color': '#7f8c8d'}),
            html.H2(id='kpi-rsi', style={'margin': '10px 0', 'color': '#2c3e50'})
        ]),
    ]),

    # Graphique Principal
    html.Div(style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '10px', 'boxShadow': '0 4px 6px rgba(0,0,0,0.1)'}, children=[
        dcc.Graph(id='graphique-principal')
    ])
])

# --- 4. LOGIQUE D'INTERACTION (Les Callbacks) ---
@app.callback(
    [Output('kpi-cours', 'children'),
     Output('kpi-volume', 'children'),
     Output('kpi-rsi', 'children'),
     Output('graphique-principal', 'figure')],
    [Input('dropdown-action', 'value')]
)
def update_dashboard(symbole_choisi):
    if not symbole_choisi or df_brvm.empty:
        return "-", "-", "-", go.Figure()

    # 1. Filtrer et calculer
    df_action = df_brvm[df_brvm['symbole'] == symbole_choisi]
    df_action = calculate_technical_indicators(df_action)
    
    dernier_cours = df_action.iloc[0]

    # 2. Préparer les textes des KPI
    variation = dernier_cours['variation']
    var_text = f" ({variation*100:+.2f}%)" if pd.notnull(variation) else ""
    txt_cours = f"{dernier_cours['cours_cloture']} FCFA{var_text}"
    txt_volume = f"{dernier_cours['volume']:,.0f}".replace(",", " ")
    
    rsi_val = dernier_cours['RSI_14']
    txt_rsi = f"{rsi_val:.1f}" if pd.notnull(rsi_val) else "N/A"
    if pd.notnull(rsi_val):
        txt_rsi += " 🔴" if rsi_val > 70 else " 🟢" if rsi_val < 30 else " ⚪"

    # 3. Construire le graphique Plotly
    fig = go.Figure()
    
    # Prix
    fig.add_trace(go.Scatter(x=df_action['date_refresh'], y=df_action['cours_cloture'], 
                             mode='lines', name='Cours de Clôture', line=dict(color='#2980b9', width=2)))
    # SMA 20
    fig.add_trace(go.Scatter(x=df_action['date_refresh'], y=df_action['SMA_20'], 
                             mode='lines', name='SMA 20', line=dict(color='#f39c12', dash='dot')))
    # SMA 50
    fig.add_trace(go.Scatter(x=df_action['date_refresh'], y=df_action['SMA_50'], 
                             mode='lines', name='SMA 50', line=dict(color='#e74c3c', dash='dash')))

    fig.update_layout(
        title=f"Analyse Technique : {symbole_choisi}",
        xaxis_title="Date",
        yaxis_title="Prix (FCFA)",
        hovermode="x unified",
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(l=40, r=20, t=40, b=40)
    )
    # Ajout d'une grille légère
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#ecf0f1')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#ecf0f1')

    # Retourner les valeurs pour mettre à jour la page
    return txt_cours, txt_volume, txt_rsi, fig

In [1]:
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output
#import sys
#import os
#sys.path.append('/Users/aemmanuelhounkponou/Developer/Personal/Invest_brvm')
from database import get_supabase_client

# --- 1. FONCTIONS DATA & SCIENCE ---
def load_data():
    """Charge TOUTES les données depuis Supabase en contournant la limite de 1000 lignes."""
    supabase = get_supabase_client()
    all_data = []
    
    # vu que supabase autorise max 1000 par défaut
    page_size = 1000 
    offset = 0
    
    print("⏳ Téléchargement de l'historique en cours...")
    
    while True:
        # On demande les lignes de 'offset' à 'offset + 999'
        response = supabase.table("full_stock_history") \
            .select("*") \
            .range(offset, offset + page_size - 1) \
            .execute()
            
        data = response.data
        
        # S'il n'y a plus de données, on casse la boucle
        if not data:
            break
            
        all_data.extend(data)
        offset += page_size
        
        # Si on récupère moins de 1000 lignes, c'est qu'on a atteint la toute fin
        if len(data) < page_size:
            break

    print(f"✅ {len(all_data)} lignes téléchargées avec succès !")
    
    # On transforme le tout en DataFrame
    df = pd.DataFrame(all_data)
    if not df.empty:
        df['date'] = pd.to_datetime(df['date'])
        
    return df


def calculate_technical_indicators(df):
    """Calcule les indicateurs techniques pour chaque action séparément."""
    # 1. On trie par Action (symbole) PUIS par Date chronologique
    df = df.sort_values(by=['symbole', 'date'], ascending=[True, True]).copy()
    
    # 2. On calcule les moyennes mobiles PAR GROUPE (par action)
    df['SMA_20'] = df.groupby('symbole')['close'].transform(lambda x: x.rolling(window=20).mean())
    df['SMA_50'] = df.groupby('symbole')['close'].transform(lambda x: x.rolling(window=50).mean())
    
    # 3. On calcule le RSI PAR GROUPE
    delta = df.groupby('symbole')['close'].diff()
    gain = (delta.where(delta > 0, 0)).groupby(df['symbole']).transform(lambda x: x.rolling(window=14).mean())
    loss = (-delta.where(delta < 0, 0)).groupby(df['symbole']).transform(lambda x: x.rolling(window=14).mean())
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    
    # On remet du plus récent au plus ancien pour l'affichage
    return df.sort_values(by='date', ascending=False)

# Chargement initial des données
df_brvm = load_data()
df_brvm1 = calculate_technical_indicators(df_brvm)
symboles_disponibles = sorted(df_brvm1['symbole'].unique()) if not df_brvm1.empty else []
print(f"Symboles disponibles pour le dashboard : {symboles_disponibles[:5]}...")  # Affiche les 5 premiers symboles pour vérification

# --- 5. LANCEMENT DU SERVEUR ---
#if __name__ == '__main__':
#    app.run_server(debug=True)



⏳ Téléchargement de l'historique en cours...
✅ 154384 lignes téléchargées avec succès !
Symboles disponibles pour le dashboard : ['ABJC', 'BICB', 'BICC', 'BNBC', 'BOAB']...


In [11]:
df_brvm[df_brvm['symbole']=='ABJC']['date'].unique()

<DatetimeArray>
['2026-05-13 00:00:00', '2026-05-12 00:00:00', '2026-05-11 00:00:00',
 '2026-05-08 00:00:00', '2026-05-07 00:00:00', '2026-05-06 00:00:00',
 '2026-05-05 00:00:00', '2026-05-04 00:00:00', '2026-04-30 00:00:00',
 '2026-04-29 00:00:00',
 ...
 '2000-04-21 00:00:00', '2000-04-19 00:00:00', '2000-04-17 00:00:00',
 '2000-04-14 00:00:00', '2000-04-12 00:00:00', '2000-04-10 00:00:00',
 '2000-04-07 00:00:00', '2000-04-05 00:00:00', '2000-04-03 00:00:00',
 '2000-03-31 00:00:00']
Length: 3736, dtype: datetime64[us]

In [7]:
d=load_data()
d[d['symbole']=='ABJC']['date'].unique()

<StringArray>
['2026-05-13', '2026-05-12', '2026-05-11', '2026-05-08', '2026-05-07',
 '2026-05-06', '2026-05-05', '2026-05-04', '2026-04-30', '2026-04-29',
 '2026-04-28', '2026-04-27', '2026-04-24', '2026-04-23', '2026-04-22',
 '2026-04-21', '2026-04-20', '2026-04-17', '2026-04-16', '2026-04-15',
 '2026-04-14', '2026-04-13']
Length: 22, dtype: str

In [8]:
d.date.unique()

<StringArray>
['2026-05-13', '2026-05-12', '2026-05-11', '2026-05-08', '2026-05-07',
 '2026-05-06', '2026-05-05', '2026-05-04', '2026-04-30', '2026-04-29',
 '2026-04-28', '2026-04-27', '2026-04-24', '2026-04-23', '2026-04-22',
 '2026-04-21', '2026-04-20', '2026-04-17', '2026-04-16', '2026-04-15',
 '2026-04-14', '2026-04-13']
Length: 22, dtype: str

In [3]:
df_brvm1[df_brvm1['symbole']=='BOABF']

,symbole,nom,volume,previous_close,open,high,low,close,variation,valeur_echangee,per,pourcentage_valeur,date,SMA_20,SMA_50,RSI_14
5,BOABF,BANK OF AFRICA BURKINA FASO,2437,5410.0,5400,5405,5405,5405,-0.09,13169135,12.36,1.01,2026-05-14,5475.50,5391.6,50.485437
52,BOABF,BANK OF AFRICA BURKINA FASO,2437,5410.0,5400,5410,5405,5405,-0.09,13169135,12.36,1.01,2026-05-13,5486.75,5379.2,48.598131
99,BOABF,BANK OF AFRICA BURKINA FASO,5219,5410.0,5400,5410,5410,5410,0.00,28188865,12.36,1.92,2026-05-12,5498.75,5366.9,35.135135
146,BOABF,BANK OF AFRICA BURKINA FASO,3605,5400.0,5400,5435,5400,5410,0.19,19503050,12.36,1.16,2026-05-11,5512.25,5352.7,34.437086
193,BOABF,BANK OF AFRICA BURKINA FASO,1414,5400.0,5405,5435,5400,5400,0.00,7635600,12.34,0.66,2026-05-08,5525.75,5337.7,34.868421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124376,BOABF,BANK OF AFRICA BURKINA FASO,26921,737.0,745,745,745,745,1.09,20056145,1.70,8.63,2011-01-07,NaN,NaN,NaN
124397,BOABF,BANK OF AFRICA BURKINA FASO,32321,750.0,737,737,737,737,-1.73,23820577,1.68,25.09,2011-01-06,NaN,NaN,NaN
124412,BOABF,BANK OF AFRICA BURKINA FASO,97523,750.0,750,750,750,750,0.00,73142250,1.71,48.38,2011-01-05,NaN,NaN,NaN
124429,BOABF,BANK OF AFRICA BURKINA FASO,38641,714.0,750,750,750,750,5.04,28980750,1.71,12.97,2011-01-04,NaN,NaN,NaN
